# Vision-Based Defect Detection Walkthrough
### End-to-End Industrial Quality Inspection Pipeline
This interactive notebook demonstrates each stage of the Vision-Based Defect Detection system:
1. **Dataset Scaffolding & Synthetic Defect Injection**
2. **OpenCV Preprocessing & Lighting Normalization (CLAHE & Bilateral)**
3. **Convolutional Autoencoder Anomaly Detection & Thresholding**
4. **Multi-Class Defect Classification (EfficientNet-B0)**
5. **Grad-CAM Localization & OpenCV Contour Metric Extraction**
6. **Refinement, Test-Time Augmentation (TTA), and Borderline Quarantine**

In [ ]:
import os
import sys
sys.path.insert(0, '../src')

import cv2
import numpy as np
import matplotlib.pyplot as plt

from dataset import prepare_dataset, create_base_texture, inject_crack, inject_scratch, inject_dent
from preprocessing import preprocess_image, apply_clahe_lab, denoise_bilateral
from anomaly_detection import AnomalyDetector
from defect_classifier import DefectClassifier
from localization import extract_defect_regions, overlay_localization_result
from refinement import RefinementEngine

print("All modules loaded successfully!")

## 1. Synthetic Defect Generation
Generate sample substrates and inject realistic procedural flaws.

In [ ]:
base_metal = create_base_texture(256, 256, 'brushed_metal')
crack_img, crack_mask = inject_crack(base_metal)
scratch_img, scratch_mask = inject_scratch(base_metal)
dent_img, dent_mask = inject_dent(base_metal)

fig, axes = plt.subplots(2, 3, figsize=(12, 8))
axes[0, 0].imshow(cv2.cvtColor(crack_img, cv2.COLOR_BGR2RGB)); axes[0, 0].set_title('Crack Defect')
axes[1, 0].imshow(crack_mask, cmap='gray'); axes[1, 0].set_title('Crack Ground Truth Mask')
axes[0, 1].imshow(cv2.cvtColor(scratch_img, cv2.COLOR_BGR2RGB)); axes[0, 1].set_title('Scratch Defect')
axes[1, 1].imshow(scratch_mask, cmap='gray'); axes[1, 1].set_title('Scratch Ground Truth Mask')
axes[0, 2].imshow(cv2.cvtColor(dent_img, cv2.COLOR_BGR2RGB)); axes[0, 2].set_title('Dent Defect')
axes[1, 2].imshow(dent_mask, cmap='gray'); axes[1, 2].set_title('Dent Ground Truth Mask')
for ax in axes.flat: ax.axis('off')
plt.tight_layout()
plt.show()

## 2. Preprocessing & CLAHE (LAB Space)
Test luminance-channel CLAHE and bilateral edge-preserving filter.

In [ ]:
final_processed, intermediates = preprocess_image(crack_img)
fig, axes = plt.subplots(1, 4, figsize=(16, 4))
axes[0].imshow(cv2.cvtColor(crack_img, cv2.COLOR_BGR2RGB)); axes[0].set_title('Original Input')
axes[1].imshow(cv2.cvtColor(intermediates.get('clahe', crack_img), cv2.COLOR_BGR2RGB)); axes[1].set_title('CLAHE (L-Channel)')
axes[2].imshow(cv2.cvtColor(intermediates.get('denoised', crack_img), cv2.COLOR_BGR2RGB)); axes[2].set_title('Bilateral Denoised')
axes[3].imshow(cv2.cvtColor(final_processed, cv2.COLOR_BGR2RGB)); axes[3].set_title('Final Letterboxed')
for ax in axes: ax.axis('off')
plt.tight_layout()
plt.show()

## 3. End-to-End Refined Inspection Pipeline

In [ ]:
detector = AnomalyDetector(approach='autoencoder')
classifier = DefectClassifier(backbone='efficientnet_b0')
engine = RefinementEngine(detector, classifier)

result = engine.inspect(final_processed, sample_id='DEMO_BATCH_001')
print(f"Decision: {result.decision}")
print(f"Defect Type: {result.defect_type}")
print(f"Confidence: {result.overall_confidence:.2%}")
print(f"Borderline Flag: {result.is_borderline}")
print(f"Detected Regions: {len(result.cleaned_regions)}")

plt.figure(figsize=(6, 6))
plt.imshow(cv2.cvtColor(result.annotated_image, cv2.COLOR_BGR2RGB))
plt.title(f"{result.decision}: {result.defect_type} ({result.overall_confidence:.1%})")
plt.axis('off')
plt.show()